# 02 - Cleaning and star schema prototype
Prototype of every cleaning rule in pandas, so the Power Query version can be checked against it.
Output: tidy star-schema CSVs in `../processed/` (also usable as a fallback Power BI source).

**Decisions taken from notebook 01**
| Finding | Rule |
|---|---|
| Artist attributes in Artworks are positional `(x) (y)` lists that align with ConstituentID | Don't parse them. Use **Artists.csv** as the single artist source and a bridge table |
| `BeginDate`/`EndDate` = 0 | Means unknown / living. Set to null |
| Blank Nationality / Gender (21% / 16% of artists) | Label `Unknown` |
| Gender has 7 values | Standardise to Male / Female / Non-binary & other / Unknown |
| `Date` is free text, 97% have a 4-digit year | `YearCreated` = first 4-digit year; `DateQuality` flag |
| `DateAcquired` blank 4.3% | Null, unknown-date member in DimDate |
| Dimension 0 = unknown (Depth 10k zeros) | 0 -> null; drop Seat Height (100% blank) and Circumference (10 values) |
| OnView xlsx is a strict subset of CSV OnView (1,142 vs 1,210, locations identical) | Use CSV as source; xlsx used as validation |
| 1,269 works have no artist | Keep in fact, `HasArtist = False`, no bridge rows |

In [1]:
import re, pandas as pd, numpy as np
from pathlib import Path
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
RAW, OUT = Path("../raw_data"), Path("../processed"); OUT.mkdir(exist_ok=True)
aw = pd.read_csv(RAW/"Artworks.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
ar = pd.read_csv(RAW/"Artists.csv",  dtype=str, keep_default_na=False, encoding="utf-8-sig")
ov_aw = pd.read_excel(RAW/"MoMA_OnView.xlsx", sheet_name="Artworks", dtype=str, keep_default_na=False)
blank = lambda s: s.str.strip().replace("", np.nan)

## DimArtist

In [2]:
GENDER = {"male": "Male", "female": "Female", "female (transwoman)": "Non-binary & other", "transgender woman": "Non-binary & other",
          "non-binary": "Non-binary & other", "gender non-conforming": "Non-binary & other"}
dim_artist = pd.DataFrame({
    "ArtistKey":    ar.ConstituentID.astype(int),
    "ArtistName":   ar.DisplayName.str.strip(),
    "ArtistBio":    blank(ar.ArtistBio),
    "Nationality":  blank(ar.Nationality).fillna("Unknown"),
    "Gender":       blank(ar.Gender).str.lower().map(GENDER).fillna("Unknown"),
    "BirthYear":    pd.to_numeric(ar.BeginDate).replace(0, np.nan),
    "DeathYear":    pd.to_numeric(ar.EndDate).replace(0, np.nan),
    "HasWikidata":  ar["Wiki QID"].str.strip() != "",
    "HasULAN":      ar["ULAN"].str.strip() != "",
})
dim_artist["BirthDecade"] = (dim_artist.BirthYear // 10 * 10).astype("Int64")
dim_artist["LifeStatus"] = np.where(dim_artist.DeathYear.notna(), "Deceased", np.where(dim_artist.BirthYear.notna(), "Living / death unknown", "Unknown"))
# nationality long tail -> group for visuals; keep the full value too
top = dim_artist.Nationality.value_counts().head(15).index
dim_artist["NationalityGroup"] = np.where(dim_artist.Nationality.isin(top) | (dim_artist.Nationality == "Unknown"), dim_artist.Nationality, "Other")
# add an 'Unknown artist' member so works without artists can still relate (key 0)
dim_artist = pd.concat([pd.DataFrame([{"ArtistKey": 0, "ArtistName": "Unknown / no artist", "Nationality": "Unknown", "Gender": "Unknown", "NationalityGroup": "Unknown", "LifeStatus": "Unknown", "HasWikidata": False, "HasULAN": False}]), dim_artist], ignore_index=True)
print(dim_artist.shape); print(dim_artist.Gender.value_counts()); dim_artist.head(3)

(15639, 12)
Gender
Male                  9940
Unknown               3258
Female                2435
Non-binary & other       6
Name: count, dtype: int64


,ArtistKey,ArtistName,Nationality,Gender,NationalityGroup,LifeStatus,HasWikidata,HasULAN,ArtistBio,BirthYear,DeathYear,BirthDecade
0,0,Unknown / no artist,Unknown,Unknown,Unknown,Unknown,False,False,NaN,NaN,NaN,<NA>
1,1,Robert Arneson,American,Male,American,Deceased,False,False,"American, 1930–1992",1930.0,1992.0,1930
2,2,Doroteo Arnaiz,Spanish,Male,Spanish,Living / death unknown,False,False,"Spanish, born 1936",1936.0,NaN,1930


## Bridge: artwork <-> artist

In [3]:
b = aw[["ObjectID", "ConstituentID"]].copy()
b["ArtistKey"] = b.ConstituentID.str.split(r",\s*")
b = b.explode("ArtistKey")
b = b[b.ArtistKey.str.strip() != ""]
b["ArtistPosition"] = b.groupby("ObjectID").cumcount() + 1
bridge = pd.DataFrame({"ObjectID": b.ObjectID.astype(int), "ArtistKey": b.ArtistKey.astype(int), "ArtistPosition": b.ArtistPosition})
assert not bridge.duplicated(["ObjectID", "ArtistKey"]).any(), "an artist appears twice on a work"
assert bridge.ArtistKey.isin(dim_artist.ArtistKey).all()
print(bridge.shape); bridge.head()

(172916, 3)


,ObjectID,ArtistKey,ArtistPosition
0,2,6210,1
1,3,7470,1
2,4,7605,1
3,5,7056,1
4,6,7605,1


## DimClassification, DimDepartment, DimMediumGroup

In [4]:
dim_dept = pd.DataFrame({"Department": sorted(aw.Department.unique())}); dim_dept.insert(0, "DepartmentKey", range(1, len(dim_dept)+1))
cls = aw.Classification.replace({"(not assigned)": "Unassigned", "": "Unassigned"})
dim_class = pd.DataFrame({"Classification": sorted(cls.unique())}); dim_class.insert(0, "ClassificationKey", range(1, len(dim_class)+1))
# group the 39 classifications into a handful of reporting groups
GRP = {"Photograph": "Photography", "Photography Research/Reference": "Photography",
       "Print": "Prints & Books", "Illustrated Book": "Prints & Books", "Periodical": "Prints & Books", "Poster": "Prints & Books", "Publication": "Prints & Books", "Multiple": "Prints & Books", "Ephemera": "Prints & Books", "Notebook": "Prints & Books",
       "Drawing": "Drawings & Paintings", "Painting": "Drawings & Paintings", "Work on Paper": "Drawings & Paintings", "Collage": "Drawings & Paintings",
       "Mies van der Rohe Archive": "Architecture & Design", "Frank Lloyd Wright Archive": "Architecture & Design", "Architecture": "Architecture & Design", "Design": "Architecture & Design", "Architectural Model": "Architecture & Design", "Furniture and Interiors": "Architecture & Design", "Graphic Design": "Architecture & Design", "Textile": "Architecture & Design", "Wallpaper": "Architecture & Design", "Fashion": "Architecture & Design",
       "Video": "Time-based media", "Film": "Time-based media", "Film (object)": "Time-based media", "Audio": "Time-based media", "Media": "Time-based media", "Moving Image": "Time-based media", "Performance": "Time-based media", "Installation": "Time-based media", "Software": "Time-based media", "Digital": "Time-based media",
       "Sculpture": "Sculpture"}
dim_class["ClassificationGroup"] = dim_class.Classification.map(GRP).fillna("Other")
dim_class.loc[dim_class.Classification == "Unassigned", "ClassificationGroup"] = "Unassigned"
print(dim_class[dim_class.ClassificationGroup == "Other"])
dim_class.head()

    ClassificationKey  Classification ClassificationGroup
4                   5  Correspondence               Other
7                   8        Document               Other
22                 23   News Clipping               Other


,ClassificationKey,Classification,ClassificationGroup
0,1,Architectural Model,Architecture & Design
1,2,Architecture,Architecture & Design
2,3,Audio,Time-based media
3,4,Collage,Drawings & Paintings
4,5,Correspondence,Other


In [5]:
MEDIUM_RULES = [  # ordered: first match wins
 ("Photography",  r"gelatin silver|albumen|chromogenic|dye transfer|daguerreo|cyanotype|salted paper|platinum print|calotype|photograph|polaroid|c-print|silver dye|inkjet|ektacolor|collodion|tintype|ambrotype"),
 ("Painting",     r"\boil\b|acrylic|synthetic polymer|tempera|enamel|\bpaint"),
 ("Printmaking",  r"lithograph|etching|woodcut|screenprint|screen print|engraving|drypoint|aquatint|linocut|letterpress|offset|photogravure|monotype|mezzotint|serigraph|\bprint\b|relief|stencil|collotype|linoleum|photocopy|xerox|poster"),
 ("Books & ephemera", r"book|journal|folder|vinyl|record|portfolio|periodical|newspaper|magazine|pamphlet"),
 ("Drawing",      r"pencil|graphite|charcoal|\bink\b|crayon|pastel|watercolor|gouache|marker|chalk|pen\b"),
 ("Time-based",   r"video|film|16mm|35mm|super 8|audio|sound|digital|projection"),
 ("Sculpture / objects", r"bronze|marble|steel|wood|plaster|ceramic|glass|aluminum|plastic|metal|stone|clay|porcelain|fiberglass|textile|cotton|silk|wool"),
]
def medium_group(m):
    if pd.isna(m): return "Unknown"
    m = m.lower()
    for name, rx in MEDIUM_RULES:
        if re.search(rx, m): return name
    return "Other"
med = blank(aw.Medium)
mg = med.map(medium_group)
print(mg.value_counts(), f"\nOther share: {(mg=='Other').mean():.1%}")
print("\nsample of 'Other' mediums:"); print(med[mg == "Other"].value_counts().head(15))

Medium
Printmaking            59227
Photography            34575
Drawing                28759
Unknown                 9247
Other                   6295
Painting                6034
Books & ephemera        5325
Sculpture / objects     4998
Time-based              3170
Name: count, dtype: int64 
Other share: 4.0%

sample of 'Other' mediums:
Medium
Drawing                                        120
CD                                             112
(CONFIRM)                                      104
Various materials                               98
One from a series of eighty-two photocopies     82
Blueprint                                       72
(confirm)                                       70
drawing                                         68
Fabric                                          66
Diazotype                                       65
Photostat                                       64
Archival envelope                               62
Fabric collage                          

## DimLocation (on view)
Source: CSV `OnView` (superset). Parse floor and gallery. 'Not on view' is a real member.

In [6]:
ovs = aw.OnView.str.strip().str.strip('"').str.strip()
loc = ovs[ovs != ""].drop_duplicates()
parsed = loc.str.extract(r"^MoMA, Floor (?P<Floor>\d+)(?:, (?P<Gallery>.+))?$")
parsed.loc[loc.str.startswith("MoMA PS1"), "Floor"] = "PS1"
print("unparsed locations:"); print(loc[parsed.Floor.isna()].tolist())

unparsed locations:
[]


In [7]:
dim_loc = pd.DataFrame({"LocationRaw": loc.values, "Floor": parsed.Floor.values, "Gallery": parsed.Gallery.values})
dim_loc["Floor"] = dim_loc.Floor.fillna("Other")
dim_loc["Gallery"] = dim_loc.Gallery.fillna("General / unspecified")
dim_loc["GalleryLabel"] = np.where(dim_loc.Floor == "PS1", dim_loc.LocationRaw, "Floor " + dim_loc.Floor + " - " + dim_loc.Gallery)
dim_loc = pd.concat([pd.DataFrame([{"LocationRaw": "", "Floor": "Not on view", "Gallery": "Not on view", "GalleryLabel": "Not on view"}]), dim_loc], ignore_index=True)
dim_loc.insert(0, "LocationKey", range(0, len(dim_loc)))
loc_key = dict(zip(dim_loc.LocationRaw, dim_loc.LocationKey))
dim_loc.drop(columns="LocationRaw").head(8)

,LocationKey,Floor,Gallery,GalleryLabel
0,0,Not on view,Not on view,Not on view
1,1,4,417,Floor 4 - 417
2,2,5,511,Floor 5 - 511
3,3,1,1 South,Floor 1 - 1 South
4,4,5,516,Floor 5 - 516
5,5,3,General / unspecified,Floor 3 - General / unspecified
6,6,2,2 South,Floor 2 - 2 South
7,7,1,General / unspecified,Floor 1 - General / unspecified


## FactArtwork
One row per `ObjectID`.

In [8]:
d = aw.Date
yr = pd.to_numeric(d.str.extract(r"(\d{4})")[0], errors="coerce")
yr_end = pd.to_numeric(d.str.extract(r"^\(?\D*\d{4}\s*[^\d\s]\s*(\d{4}|\d{2})\b")[0], errors="coerce")
quality = np.select([yr.isna(), d.str.fullmatch(r"\d{4}"), d.str.contains(r"c\.|ca\.|circa|\?", case=False), d.str.contains(r"\d{4}\s*[^\d\s]\s*\d{2,4}")],
                    ["Unknown", "Exact", "Approximate", "Range"], "Approximate")
acq = pd.to_datetime(aw.DateAcquired.replace("", np.nan), errors="coerce")
num = lambda col: pd.to_numeric(aw[col], errors="coerce").replace(0, np.nan)
n_art = aw.ConstituentID.str.split(r",\s*").apply(lambda x: 0 if x == [""] else len(x))
cl = pd.Series(cls.values);
credit = aw.CreditLine.str.lower()
fact = pd.DataFrame({
    "ObjectID": aw.ObjectID.astype(int), "Title": blank(aw.Title).fillna("Untitled"), "AccessionNumber": aw.AccessionNumber,
    "DepartmentKey": aw.Department.map(dict(zip(dim_dept.Department, dim_dept.DepartmentKey))),
    "ClassificationKey": cls.map(dict(zip(dim_class.Classification, dim_class.ClassificationKey))),
    "MediumGroup": mg, "Medium": med,
    "YearCreated": yr.astype("Int64"), "DateQuality": quality, "DateText": blank(d),
    "AcquiredDate": acq, "YearAcquired": acq.dt.year.astype("Int64"),
    "ArtistCount": n_art, "HasArtist": n_art > 0, "IsMultiArtist": n_art > 1,
    "IsCataloged": aw.Cataloged.eq("Y"), "IsOnView": ovs != "", "LocationKey": ovs.map(lambda x: loc_key.get(x, 0) if x else 0),
    "AcquisitionType": np.select([credit.str.contains(r"gift|given|bequest|promised"), credit.str.contains(r"purchase|fund|acquired through")], ["Gift / bequest", "Purchase / fund"], "Other / unknown"),
    "CreditLine": blank(aw.CreditLine),
    "Height_cm": num("Height (cm)"), "Width_cm": num("Width (cm)"), "Depth_cm": num("Depth (cm)"), "Length_cm": num("Length (cm)"),
    "Diameter_cm": num("Diameter (cm)"), "Weight_kg": num("Weight (kg)"), "Duration_sec": num("Duration (sec.)"),
    "URL": blank(aw.URL),
})
fact["AgeAtAcquisition"] = (fact.YearAcquired - fact.YearCreated).astype("Int64")
fact["AgeAtAcquisition"] = fact.AgeAtAcquisition.where(fact.AgeAtAcquisition >= 0)   # negatives = date conflicts
fact["HasValidDimensions"] = fact.Height_cm.notna() & fact.Width_cm.notna()
fact["Area_cm2"] = (fact.Height_cm * fact.Width_cm).where(fact.HasValidDimensions)
print(fact.shape); print(fact.DateQuality.value_counts()); fact.head(3).T

(157630, 31)
DateQuality
Exact          97465
Range          32706
Approximate    22993
Unknown         4466
Name: count, dtype: int64


,0,1,2
ObjectID,2,3,4
Title,"Ferdinandsbrücke Project, Vienna, Austria (Ele...","City of Music, National Superior Conservatory ...","Villa project, outside Vienna, Austria (Elevat..."
AccessionNumber,885.1996,1.1995,1.1997
DepartmentKey,1,1,1
ClassificationKey,2,2,2
MediumGroup,Painting,Painting,Drawing
Medium,Ink and cut-and-pasted painted pages on paper,Paint and colored pencil on print,"Graphite, pen, color pencil, ink, and gouache ..."
YearCreated,1896,1987,1903
DateQuality,Exact,Exact,Exact
DateText,1896,1987,1903


## DimDate (acquisition calendar)

In [9]:
start, end = acq.min().replace(month=1, day=1), pd.Timestamp(acq.max().year, 12, 31)
dd = pd.DataFrame({"Date": pd.date_range(start, end)})
dd["DateKey"] = dd.Date.dt.strftime("%Y%m%d").astype(int)
dd["Year"], dd["Quarter"], dd["MonthNumber"] = dd.Date.dt.year, "Q" + dd.Date.dt.quarter.astype(str), dd.Date.dt.month
dd["MonthName"] = dd.Date.dt.strftime("%b"); dd["Decade"] = (dd.Year // 10 * 10).astype(str) + "s"
dd["Era"] = pd.cut(dd.Year, [1928, 1945, 1970, 2000, 2030], labels=["Founding 1929-45", "Post-war 1946-70", "Contemporary 1971-2000", "Modern 2001+"]).astype(str)
dim_date = dd
fact["AcquiredDateKey"] = acq.dt.strftime("%Y%m%d").astype("Int64")
print(dim_date.shape, dim_date.Date.min().date(), dim_date.Date.max().date())

(35064, 8) 1929-01-01 2024-12-31


## Validation (must all pass)

In [10]:
checks = {
 "fact rows == raw rows": len(fact) == len(aw),
 "ObjectID unique": fact.ObjectID.is_unique,
 "all DepartmentKey resolved": fact.DepartmentKey.notna().all(),
 "all ClassificationKey resolved": fact.ClassificationKey.notna().all(),
 "all LocationKey resolved": fact.LocationKey.isin(dim_loc.LocationKey).all(),
 "bridge ObjectIDs exist in fact": bridge.ObjectID.isin(fact.ObjectID).all(),
 "bridge ArtistKeys exist in DimArtist": bridge.ArtistKey.isin(dim_artist.ArtistKey).all(),
 "works with artists == distinct bridge ObjectIDs": fact.HasArtist.sum() == bridge.ObjectID.nunique(),
 "sum(ArtistCount) == bridge rows": fact.ArtistCount.sum() == len(bridge),
 "AcquiredDateKey in DimDate": fact.AcquiredDateKey.dropna().isin(dim_date.DateKey).all(),
 "IsOnView == 1,210 (CSV)": fact.IsOnView.sum() == 1210,
 "xlsx on-view works all flagged on view": fact[fact.ObjectID.isin(ov_aw.ObjectID.astype(int))].IsOnView.all(),
 "no negative dimensions": (fact[["Height_cm","Width_cm","Depth_cm"]].fillna(1) > 0).all().all(),
}
for k, v in checks.items(): print("PASS" if v else "FAIL", k)
assert all(checks.values())

PASS fact rows == raw rows
PASS ObjectID unique
PASS all DepartmentKey resolved
PASS all ClassificationKey resolved
PASS all LocationKey resolved
PASS bridge ObjectIDs exist in fact
PASS bridge ArtistKeys exist in DimArtist
PASS works with artists == distinct bridge ObjectIDs
PASS sum(ArtistCount) == bridge rows
PASS AcquiredDateKey in DimDate
PASS IsOnView == 1,210 (CSV)
PASS xlsx on-view works all flagged on view
PASS no negative dimensions


## Sanity-check KPI answers (to compare later against the Power BI measures)

In [11]:
bw = bridge.merge(dim_artist[["ArtistKey","Gender","Nationality"]], on="ArtistKey").merge(fact[["ObjectID","YearAcquired"]], on="ObjectID")
print(f"Works: {len(fact)} | Artists with >=1 work: {bridge.ArtistKey.nunique()} | cataloged: {fact.IsCataloged.mean():.1%} | on view: {fact.IsOnView.mean():.2%}")
print("\nArtist-credits by gender (bridge rows, known gender only):")
print(bw[bw.Gender != "Unknown"].Gender.value_counts(normalize=True).round(3))
print("\nWorks by acquisition decade:"); print((fact.YearAcquired // 10 * 10).value_counts().sort_index())
print("\nOn-view rate by department:")
print(fact.groupby("DepartmentKey").IsOnView.mean().mul(100).round(2).rename(index=dict(zip(dim_dept.DepartmentKey, dim_dept.Department))))
print("\nTop 10 nationalities by artist-credits:"); print(bw.Nationality.value_counts().head(10))

Works: 157630 | Artists with >=1 work: 14423 | cataloged: 63.6% | on view: 0.77%

Artist-credits by gender (bridge rows, known gender only):
Gender
Male                  0.854
Female                0.146
Non-binary & other    0.000
Name: proportion, dtype: float64

Works by acquisition decade:
YearAcquired
1920        9
1930     1814
1940     7946
1950     6406
1960    43741
1970    13291
1980    10213
1990    11488
2000    24834
2010    25071
2020     6068
Name: count, dtype: Int64

On-view rate by department:
DepartmentKey
Architecture & Design                     0.59
Architecture & Design - Image Archive     0.00
Drawings & Prints                         0.35
Film                                      1.37
Fluxus Collection                         0.00
Media and Performance                     0.78
Painting & Sculpture                     11.49
Photography                               0.68
Name: IsOnView, dtype: float64

Top 10 nationalities by artist-credits:
Nationality
American 

## Export
Column names are final Power BI names. Dates as ISO, booleans as TRUE/FALSE, UTF-8.

In [12]:
dim_artist.drop(columns=[], errors="ignore").to_csv(OUT/"DimArtist.csv", index=False, encoding="utf-8-sig")
dim_class.to_csv(OUT/"DimClassification.csv", index=False, encoding="utf-8-sig")
dim_dept.to_csv(OUT/"DimDepartment.csv", index=False, encoding="utf-8-sig")
dim_loc.drop(columns="LocationRaw").to_csv(OUT/"DimLocation.csv", index=False, encoding="utf-8-sig")
dim_date.to_csv(OUT/"DimDate.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
bridge.to_csv(OUT/"BridgeArtworkArtist.csv", index=False, encoding="utf-8-sig")
fact.to_csv(OUT/"FactArtwork.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
for f in sorted(OUT.glob("*.csv")): print(f"{f.name:28s} {f.stat().st_size/1e6:7.1f} MB")

BridgeArtworkArtist.csv          2.5 MB
DimArtist.csv                    1.6 MB
DimClassification.csv            0.0 MB
DimDate.csv                      2.1 MB
DimDepartment.csv                0.0 MB
DimLocation.csv                  0.0 MB
FactArtwork.csv                 47.8 MB
